In [1]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# Freeze pretrained encoder weights (feature extraction)
for name, param in model.named_parameters():
    if "classifier" not in name and "pre_classifier" not in name:
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable parameters: 592,130 / 66,955,010 (0.88%)


In [1]:
import hashlib
from datasets import Dataset
from transformers import AutoTokenizer

raw_records = [
    {"instruction": "Explain LoRA rank r.", "input": "", "output": "Rank r defines the inner dimension of low-rank matrices A and B, controlling trainable capacity."},
    {"instruction": "Explain LoRA rank r.", "input": "", "output": "Rank r defines the inner dimension of low-rank matrices A and B, controlling trainable capacity."},
    {"instruction": "Write a note.", "input": "", "output": "OK"}  # Filtered out (too short)
]

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
seen_hashes, cleaned = set(), []

for row in raw_records:
    prompt_key = (row["instruction"].strip() + " " + row["input"].strip()).strip()
    h = hashlib.sha256(prompt_key.encode("utf-8")).hexdigest()
    if h not in seen_hashes and len(row["output"].split()) >= 5:
        seen_hashes.add(h)
        cleaned.append({
            "messages": [
                {"role": "system", "content": "You are a precise machine learning research assistant."},
                {"role": "user", "content": prompt_key},
                {"role": "assistant", "content": row["output"].strip()},
            ]
        })

def format_and_count(ex):
    formatted_text = tokenizer.apply_chat_template(ex["messages"], tokenize=False)
    return {
        "formatted_text": formatted_text,
        "num_tokens": len(tokenizer(formatted_text)["input_ids"]),
    }

ds = Dataset.from_list(cleaned)
ds = ds.map(format_and_count)
ds = ds.filter(lambda ex: ex["num_tokens"] <= 2048)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1 [00:00<?, ? examples/s]

In [3]:
!pip install -U torchao peft


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 20.2 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [4]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, config)
model.print_trainable_parameters()   # shows trainable params and percentage

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

W1110 18:55:13.613000 2298 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [6]:
!pip install -U bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 50.0 MB/s  0:00:00


In [8]:
!pip install -U "bitsandbytes>=0.46.1"



In [10]:
!pip install -U "bitsandbytes>=0.46.1" accelerate


In [20]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    quantization_config=bnb_config,
    device_map="auto",
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [19]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Load in 8-bit
model_8bit = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    quantization_config=BitsAndBytesConfig(load_in_8bit=True),
    device_map="auto",
)

# Load in 4-bit (NF4)
model_4bit = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    ),
    device_map="auto",
)

print(model_4bit.get_memory_footprint() / 1e9, "GB")

('TinyLlama/TinyLlama-1.1B-Chat-v1.0',)

In [21]:
def estimate_vram_gb(params_b=7.0, layers=32, kv_heads=8, head_dim=128, seq_len=4096, batch=1):
    n = params_b * 1e9
    kv_bytes = 2 * batch * seq_len * layers * kv_heads * head_dim * 2  # BF16 KV cache
    inf_bf16 = (n * 2 + kv_bytes) / 1e9 + 1.5  # +1.5 GB CUDA buffer
    fft_bf16 = (n * 16) / 1e9 + 4.0            # 16 B/param static + ~4 GB activations
    lora_16bit = (n * 2 + (0.015 * n) * 14) / 1e9 + 2.5     # 16-bit base + 1.5% LoRA params
    qlora_4bit = (n * 0.55 + (0.015 * n) * 14) / 1e9 + 2.5  # 4-bit base + 1.5% LoRA params
    return {
        "Inference_BF16_GB": round(inf_bf16, 1),
        "FFT_Static_GB": round(fft_bf16, 1),
        "LoRA_16bit_GB": round(lora_16bit, 1),
        "QLoRA_GB": round(qlora_4bit, 1),
    }

print("Approximate 7B VRAM Estimates (GB):", estimate_vram_gb(params_b=7.0))

Approximate 7B VRAM Estimates (GB): {'Inference_BF16_GB': 16.0, 'FFT_Static_GB': 116.0, 'LoRA_16bit_GB': 18.0, 'QLoRA_GB': 7.8}


In [22]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

docs = [
    "Policy A-12: Remote work requests require manager approval and VPN MFA setup.",
    "Policy B-04: Travel reimbursements must be submitted within 14 calendar days.",
]
query = "What is the deadline for travel expense reimbursement?"

tok = AutoTokenizer.from_pretrained("BAAI/bge-small-en-v1.5")
enc = AutoModel.from_pretrained("BAAI/bge-small-en-v1.5")

def embed(texts):
    inp = tok(texts, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        out = enc(**inp).last_hidden_state[:, 0]  # CLS token pooling
    return F.normalize(out, p=2, dim=1)

doc_vecs, q_vec = embed(docs), embed([query])
scores = torch.matmul(q_vec, doc_vecs.T).squeeze(0)
best_idx = int(torch.argmax(scores))
rag_prompt = f"Context: {docs[best_idx]}\nQuestion: {query}\nAnswer based only on Context:"
print(rag_prompt)

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Context: Policy B-04: Travel reimbursements must be submitted within 14 calendar days.
Question: What is the deadline for travel expense reimbursement?
Answer based only on Context:


In [23]:
from collections import deque, defaultdict

triples = [
    ("Apex_Corp", "SUPPLIES_CHIP_TO", "Nova_Robotics", "Doc_101: Apex Corp is the sole supplier of NPU-9 chips to Nova Robotics."),
    ("Nova_Robotics", "ACQUIRED_BY", "Zenith_Holdings", "Doc_409: Zenith Holdings completed the acquisition of Nova Robotics in Q3."),
    ("Zenith_Holdings", "SANCTIONED_IN", "Jurisdiction_X", "Doc_812: Regulators in Jurisdiction X placed export restrictions on Zenith Holdings."),
]

graph = defaultdict(list)
for subj, rel, obj, chunk in triples:
    graph[subj].append((rel, obj, chunk))
    graph[obj].append((f"INVERSE_{rel}", subj, chunk))

def retrieve_k_hop_context(seed_entity, max_hops=2):
    visited, queue, context_chunks = {seed_entity}, deque([(seed_entity, 0)]), []
    while queue:
        node, hop = queue.popleft()
        if hop >= max_hops:
            continue
        for rel, neighbor, chunk in graph[node]:
            context_chunks.append(f"[Hop {hop+1}] {node} --({rel})--> {neighbor} | {chunk}")
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, hop + 1))
    return context_chunks

for line in retrieve_k_hop_context("Apex_Corp", max_hops=3):
    print(line)

[Hop 1] Apex_Corp --(SUPPLIES_CHIP_TO)--> Nova_Robotics | Doc_101: Apex Corp is the sole supplier of NPU-9 chips to Nova Robotics.
[Hop 2] Nova_Robotics --(INVERSE_SUPPLIES_CHIP_TO)--> Apex_Corp | Doc_101: Apex Corp is the sole supplier of NPU-9 chips to Nova Robotics.
[Hop 2] Nova_Robotics --(ACQUIRED_BY)--> Zenith_Holdings | Doc_409: Zenith Holdings completed the acquisition of Nova Robotics in Q3.
[Hop 3] Zenith_Holdings --(INVERSE_ACQUIRED_BY)--> Nova_Robotics | Doc_409: Zenith Holdings completed the acquisition of Nova Robotics in Q3.
[Hop 3] Zenith_Holdings --(SANCTIONED_IN)--> Jurisdiction_X | Doc_812: Regulators in Jurisdiction X placed export restrictions on Zenith Holdings.
